# 05 — Bayesian Decision Rule (Classical Pipeline)

## Theory

Given class-conditional densities p(x|ω) and equal priors P(ω_M) = P(ω_B) = 0.5, Bayes' theorem gives:

$$P(\omega_M \mid x) = \frac{p(x\mid\omega_M)}{p(x\mid\omega_M) + p(x\mid\omega_B)}$$

**MAP rule** — minimises 0-1 loss:  
predict melanoma if P(ω_M | x) > 0.5

**Cost-sensitive rule** — minimises expected cost λ(FN)·P(ω_M) + λ(FP)·P(ω_B):  
predict melanoma if λ(FP)·P(ω_B|x) < λ(FN)·P(ω_M|x)  
⟹ predict melanoma if P(ω_M|x) > λ(FP) / (λ(FP) + λ(FN)) = 1 / (1 + 5) = **1/6**

## Numerical stability

Computing exp(log p(x|ω)) directly overflows/underflows for high-dimensional features.  
With equal priors the posterior simplifies to a sigmoid of the log-likelihood ratio:

$$P(\omega_M \mid x) = \sigma\!\left(\log p(x\mid\omega_M) - \log p(x\mid\omega_B)\right) = \frac{1}{1 + e^{-(\ell_M - \ell_B)}}$$

This avoids ever exponentiating large negative log-likelihoods.

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd())
for _root in [_cwd, *_cwd.parents]:
    if (_root / "skin_lesion" / "src" / "config.py").exists():
        sys.path.insert(0, str(_root / "skin_lesion" / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay)

from config import SEED, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, COST_FN, COST_FP

## 1 — Load splits and GMMs

In [ ]:
data = np.load(PROCESSED_DIR / "splits.npz")
X_val,  y_val  = data["X_val"],  data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

gmm_benign   = joblib.load(PROCESSED_DIR / "gmm_benign.pkl")
gmm_melanoma = joblib.load(PROCESSED_DIR / "gmm_melanoma.pkl")

print(f"Val  : {len(y_val)}  samples")
print(f"Test : {len(y_test)} samples")
print(f"GMM benign   — K={gmm_benign.n_components},   cov='{gmm_benign.covariance_type}'")
print(f"GMM melanoma — K={gmm_melanoma.n_components}, cov='{gmm_melanoma.covariance_type}'")

## 2 — Posterior computation (numerically stable)

`GaussianMixture.score_samples(X)` returns log p(x|ω) (per-sample log-likelihood under the mixture).  
We form the log-likelihood ratio and pass it through the sigmoid — no exp of large negatives needed.

In [ ]:
def compute_posterior(X, gmm_mel, gmm_ben):
    """Return P(omega_M | x) for each row in X using the log-likelihood ratio sigmoid."""
    log_p_mel = gmm_mel.score_samples(X)   # log p(x | omega_M)
    log_p_ben = gmm_ben.score_samples(X)   # log p(x | omega_B)
    # sigmoid(a) = 1 / (1 + exp(-a)), clipped for safety
    log_ratio = np.clip(log_p_mel - log_p_ben, -500, 500)
    return 1.0 / (1.0 + np.exp(-log_ratio))

post_val  = compute_posterior(X_val,  gmm_melanoma, gmm_benign)
post_test = compute_posterior(X_test, gmm_melanoma, gmm_benign)

print(f"Posterior P(mel|x) — val  : mean={post_val.mean():.4f}, "
      f"min={post_val.min():.4f}, max={post_val.max():.4f}")
print(f"Posterior P(mel|x) — test : mean={post_test.mean():.4f}, "
      f"min={post_test.min():.4f}, max={post_test.max():.4f}")

## 3 — Apply decision rules and evaluate

| Rule | Threshold | Rationale |
|------|-----------|----------|
| MAP  | 0.5       | Minimises 0-1 loss (equal error costs) |
| Cost-sensitive | 1/6 | Minimises expected cost with λ(FN)=5·λ(FP); a lower threshold means we accept more FP to catch more melanomas |

In [ ]:
THRESH_MAP  = 0.5
THRESH_COST = COST_FP / (COST_FP + COST_FN)   # 1.0 / 6.0

print(f"MAP threshold       : {THRESH_MAP}")
print(f"Cost-sensitive threshold: {THRESH_COST:.6f}  (= {COST_FP}/{int(COST_FP+COST_FN)})\n")

def evaluate(y_true, posterior, threshold, rule_name, split_name):
    y_pred = (posterior >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    # cm[i,j] = predicted j, true i  (sklearn convention)
    tn, fp, fn, tp = cm.ravel()
    acc  = accuracy_score(y_true, y_pred)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # sensitivity / TPR
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0   # specificity / TNR
    cost = COST_FN * fn + COST_FP * fp
    print(f"[{split_name} | {rule_name}]")
    print(f"  Accuracy={acc:.4f}  Sensitivity={sens:.4f}  Specificity={spec:.4f}")
    print(f"  FN={fn}  FP={fp}  Total cost={cost:.0f}\n")
    return dict(rule=rule_name, split=split_name, threshold=round(threshold, 6),
                accuracy=round(acc,4), sensitivity=round(sens,4),
                specificity=round(spec,4), FN=int(fn), FP=int(fp),
                total_cost=int(cost), cm=cm)

records = []
for split_name, y_true, post in [("val",  y_val,  post_val),
                                  ("test", y_test, post_test)]:
    records.append(evaluate(y_true, post, THRESH_MAP,  "MAP",           split_name))
    records.append(evaluate(y_true, post, THRESH_COST, "cost-sensitive", split_name))

## 4 — Confusion matrices on the test set

In [ ]:
test_records = {r["rule"]: r for r in records if r["split"] == "test"}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Confusion matrices — Test set", fontsize=13)

for ax, (rule, res) in zip(axes, test_records.items()):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=res["cm"],
        display_labels=["Benign", "Melanoma"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    thresh_label = "0.5" if rule == "MAP" else "1/6 ≈ 0.167"
    ax.set_title(f"{rule} (θ = {thresh_label})\n"
                 f"Sens={res['sensitivity']:.3f}  Spec={res['specificity']:.3f}",
                 fontsize=10)

plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to: {FIGURES_DIR / 'confusion_matrices.png'}")

## 5 — Summary table

In [ ]:
# keep only the test-set rows for the saved CSV
summary_cols = ["rule", "threshold", "accuracy", "sensitivity",
                "specificity", "FN", "FP", "total_cost"]
test_df = pd.DataFrame(
    [{k: r[k] for k in summary_cols} for r in records if r["split"] == "test"]
)
print(test_df.to_string(index=False))

TABLES_DIR.mkdir(parents=True, exist_ok=True)
test_df.to_csv(TABLES_DIR / "decision_comparison.csv", index=False)
print(f"\nSaved to: {TABLES_DIR / 'decision_comparison.csv'}")

## 6 — Interpretation

**Did the cost-sensitive rule reduce FN as expected?**

By lowering the decision threshold from 0.5 to 1/6 we widen the "predict melanoma" region: any sample whose posterior P(ω_M|x) lies in the interval [1/6, 0.5) is now flipped from benign to melanoma.  
This should monotonically reduce FN (missed melanomas) and monotonically increase FP (benign cases flagged as melanoma), since the two quantities trade off along the ROC curve.

The total-cost metric (5·FN + 1·FP) tells us whether the trade-off is worthwhile under the given cost matrix.  
A clinically acceptable outcome is one where:
- **Sensitivity ≥ 0.85** — most melanomas are caught, reducing mortality risk.
- The increase in FP is manageable — each false alarm leads to a follow-up biopsy, which is costly but not dangerous.

If the cost-sensitive rule does **not** improve sensitivity meaningfully, it suggests that the GMM posteriors are poorly calibrated — the model assigns posteriors far from 1/6 to most samples, so shifting the threshold has little effect.  
This motivates examining the ROC curve and calibration diagram in a later step.